[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Simone-Alghisi/HMD-Lab/blob/master/notebooks/8_multi_agent_system.ipynb)

## 1 What is agentic AI?

##### **Agentic AI** is an artificial intelligence system that accomplishes a goal with different levels of  supervisions, ranging from autonomous to fully supervised instances. Agentic AI uses LLMs as their main component to actually generate outputs to complete tasks by calling external tools, aggregating inputs and outputs in an orchestrated graph of agent instances.

<div align="center">
  <img src="assets/Agentic_Framework.png" alt="My Image" width="800">
</div>

This Image illustrates an agentic framework designed specifically for the telecommunication domain. The Planning Agent is responsible for actually getting the commands from the user and selecting the appropriate specialized agents. Specialized agents execute actions directly or delegate subtasks to other agents. 

##### - Those agents can be useful when they need to handle requests that cannot be implemented with predefined workflows. 
#####   **Example:** Find the top 5 emerging competitors in the electric aircraft industry and summarize their latest innovations.

##### - However, they introduce unpredictability and potential errors since they have to dynamically decide their own steps rather than follow a guaranteed, fixed workflow.
##### **Example:** What is the product of 456 and 789?



## 2 Core Components of an AI Agent

<div align="center">
  <img src="assets/AI_Agent_Architecture.png" alt="My Image" width="600">
</div>

### 1. LLM Model
- Acts as the **brain** of the agent.  
- Responsible for reasoning, understanding, and generating responses.  
- Maintains context and makes decisions based on knowledge and prior interactions.

### 2. Tools
- External functions, APIs, or plugins that allow the agent to perform specific actions.  
- Extend the agent’s capabilities (e.g., web search, code execution, image generation, data analysis).

### 3. Instructions
- Define the rules, goals, and behavioral constraints of the agent.  
- Guide the agent’s tone, scope of allowed actions, and ethical boundaries.

### 4. State
- Stores the **state of the conversation**, including past messages, tool outputs, and contextual information.  
- Enables continuity across multiple user interactions.

## 3 How a Basic AI Agent Works

When a user provides an input or question, the AI Agent processes it as follows:

1. The **user query** is combined with the **system instructions** and any relevant **context or memory**.
2. The **LLM** (foundation model) interprets the request and decides whether a tool should be used or if it can answer directly.
3. If a tool is required, the model generates a **JSON output** specifying:
   * The **tool to call**
   * The **parameters** needed for that tool
4. The tool is executed, and the **tool result is returned to the model**.
5. The model then uses this tool result to **generate the final response**.

### Limitations of the Basic Agent

A basic agent still operates in a mostly **single-step** or **single-cycle** pattern:

* It can call a tool once and then produce an answer
* But it **cannot decide to perform multiple tool calls in sequence**

This limits its ability to handle **multi-step reasoning** or workflows where the output of one tool must feed into another.

To overcome this limitation, a more advanced architecture was introduced: the **ReAct Framework**.


## 4 The ReAct Framework

The **ReAct Framework** (Reason + Act) extends the capabilities of AI Agents by integrating **iterative reasoning** and **tool use** in a looped process.

- **Reasoning (Reason)**:  
  The model first reflects on the problem and determines the steps needed to solve it.  
  This encourages **step-by-step logical planning** before any action is taken.

- **Action (Act)**:  
  Based on its reasoning, the model selects and executes the appropriate tools.  
  The results of these actions are then fed back into the reasoning loop.

This iterative process continues until the agent produces a **final answer**.

## 5 How a ReAct AI Agent Works

<div align="center">
  <img src="assets/ReAct_Architecture.png" alt="My Image" width="500">
</div>


Below is the general workflow of a ReAct-based AI Agent:

1. **Initialization**
   - The system prompt (instructions), user query, and any prior observations (results from earlier tool executions) are concatenated into a single input prompt.

2. **Reasoning and Action Generation**
   - The LLM receives this prompt and outputs two components sequentially:
     1. **Thought**: A reasoning step describing how the model plans to approach the problem.
     2. **Action**: A JSON-formatted instruction specifying which tool to call and with what parameters.

3. **Tool Execution**
   - The system parses the generated action, executes the corresponding tool, and stores the tool’s result as a new **observation**.

4. **Looping Process**
   - The updated prompt (including the new observation) is passed back to the LLM.  
   - This loop continues until the model produces an action of type **`final_answer`**, signaling that the reasoning process is complete.

## 6 Example: Population Comparison Task

**System prompt:**  
> "You are an agent that can browse the web and call tools. Use reasoning and then act."

**User query:**  
> "Find the current population of Venice and compare it to that of Trento."


### Step-by-Step Execution

#### 1. Initialization
   - The system stores the user’s query as a `TaskStep`.

#### 2. First Iteration
   - **Thought:** “To compare populations, I first need the population of Venice, then that of Trento, and finally compute their ratio.”  
   - **Action:**  
     ```json
     {"tool": "search_engine", "query": "population of Venice Italy 2025"}
     ```
   - **Observation:** “Venice population: 254,000 (2024 estimate).”

#### 3. Second Iteration
   - **Thought:** “Now I need the population of Trento.”  
   - **Action:**  
     ```json
     {"tool": "search_engine", "query": "population of Trento Italy 2025"}
     ```
   - **Observation:** “Trento population: 118,000 (2023 estimate).”

#### 4. Third Iteration
   - **Thought:** “I have both values; now compute the ratio.”  
   - **Action:**  
     ```json
     {"tool": "calculator", "expression": "118000 / 254000"}
     ```
   - **Observation:** “Result: 0.46.”

#### 5. Final Iteration
   - **Thought:** “I now have the answer.”  
   - **Action:**  
     ```json
     {"tool": "final_answer", "output": "The population of Trento is approximately 46% of that of Venice."}
     ```

## 7 Introduction to Smolagents

`smolagents` is one of the simplest and most lightweight frameworks for building *ReAct-style AI Agents*.  
It supports multiple Large Language Models (LLMs), including models from:

- Hugging Face Hub  
- OpenAI  
- Anthropic  
- And others

The library implements the **ReAct framework** and allows you to:

- Define custom tools
- Use pre-built tools
- Build multi-agent systems where agents collaborate on tasks

`smolagents` provides two main agent types:

- **ToolCallingAgent**: uses JSON/text-structured tool calls  
- **CodeAgent** — writes Python code as its actions and executes it in a sandbox

### 7.1 Installation

In [ ]:
%pip install "smolagents[toolkit]"

from huggingface_hub import login
login()

: 

### 7.2 ToolCallingAgent

The `ToolCallingAgent` provides a direct implementation of the ReAct framework.
It generates:

1. A **reasoning step** (“Thought”)
2. An **action step**, expressed as a JSON tool call

It then selects and executes tools based on the reasoning produced by the LLM.

### 7.3 CodeAgent

Unlike the `ToolCallingAgent`, which produces JSON tool calls,
the **CodeAgent writes Python code** as its action.
This code may:

* Call custom tools
* Perform arbitrary computations
* Use Python’s built-in capabilities

All code is executed in a **sandboxed environment**, and anything printed (`print(...)`) is treated as an **observation** and fed back into the ReAct loop.

#### Example: ToolCallingAgent vs. CodeAgent Actions

**ToolCallingAgent Action (declares a tool call)**

```json
{
  "tool_call": {
    "name": "set_temperature",
    "arguments": {
      "room": "degrees_celsius"
    }
  }
}
```

**CodeAgent Action (executes code directly)**

```python
result = set_temperature("degree_celsius")
print(result)
final_answer(result)
```

#### CodeAgent Example: Home Assistant Agent

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, DuckDuckGoSearchTool, tool

temperature = 22  # Initial temperature setting
light_status = { # Initial light status for each room
    "living room": "off",
    "kitchen": "off",
    "bedroom": "off"
}

@tool
def set_temperature(degrees_celsius: float) -> str:
    """
    Sets the thermostat to the specified temperature.

    Args:
        degrees_celsius: The desired temperature in Celsius.
    """
    if degrees_celsius < 10 or degrees_celsius > 30:
        raise ValueError("Temperature must be between 10 and 30 degrees Celsius.")
    global temperature
    temperature = degrees_celsius
    return f"The thermostat has been set to {degrees_celsius}°C."

model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")

home_assistant_agent = CodeAgent(
    tools=[set_temperature, DuckDuckGoSearchTool()],
    model=model,
    additional_authorized_imports=[],
    name="home_assistant_agent",
    description="A home assistant that can manage home devices.",
)


home_assistant_agent.run("What is the weather like today in New York?")
home_assistant_agent.run("Set the temperature to 25 degrees Celsius.")
print(f"Current temperature setting: {temperature}°C")

#### Excercise 1: 

- Add a tool to the home assistant agent that can turn on and off the light in a specified room. 

- We assume that the house has only three rooms: **living room, kitchen, and bedroom**.

### 7.4 Multi-Agent Systems in Smolagents

A multi-agent system consists of **multiple AI agents**,
each with:

* A dedicated role
* A specific set of tools
* A specialized responsibility

This design increases:

* **Modularity**
* **Scalability**
* **Robustness**

Instead of giving one massive agent too many capabilities,
tasks are distributed across specialized sub-agents.

### 7.5 Multi-Agent Architecture

A common pattern is:

* An **orchestrator agent** 
* Several **specialized sub-agents**, such as:

  * Web search agent
  * Home assistant agent
  * Calendar agent

The orchestrator delegates tasks to the appropriate agents.

#### Multi-Agent System Example: Assistant System

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, WebSearchTool, UserInputTool

model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")

calendar = [{"event": "Doctor's appointment", "date": "2025-12-10", "time": "10:00 AM"}, 
            {"event": "Team meeting", "date": "2025-12-12", "time": "2:00 PM"}]

web_agent = CodeAgent(
   tools=[DuckDuckGoSearchTool()],
   model=model,
   name="web_search_agent",
   description="Runs web searches for you. Provide a query."
)

manager_agent = CodeAgent(
   tools=[],
   model=model,
   managed_agents=[web_agent, home_assistant_agent],
   name="manager_agent",
   description="Manages multiple specialized agents to assist with various tasks.",
)

manager_agent.run("Who is the CEO of Hugging Face?")

#### Exercise 2:

- Add a calendar_agent capable of managing various calendar events.
- Ensure the agent uses and maintains the exact event format already established.
- The agent should be able to create new events and display them within the calendar view.

### 7.7 Problems

1. **AI-Agents do not truly plan:** They only output a “thought” message before generating an action or code, but this does not necessarily mean they are actually planning or reasoning.

2. **AI agents attempt tasks they cannot perform:** They try to execute actions they are not capable of handling.

3. **Lack of fallback policies:** When executing tool calls, if the model generates an incorrect json, errors occur because no fallback or recovery mechanisms exist.

4. **Potential infinite loops:** As seen in some code executions or tool-calling sequences, ReAct-style loops of “thought, action, observation” can continue indefinitely.

5. **Difficulty handling complex tasks:** How can these systems manage complex tasks? Can they genuinely reason?